---
title: "À quelques kilomètres près : lycées et géographie des cadres dans trois aires urbaines de province"
output-file: index.html
---

In [ ]:
# Le SSPCloud définit MPLBACKEND=Agg : sans cette ligne, les figures ne s'affichent pas
%matplotlib inline
import warnings

import pandas as pd
from IPython.display import display

from cadres_lycees import (
    analyse,
    cartes,
    collecte,
    config,
    figures,
    geo,
    nettoyage,
    pipeline,
    tableaux,
)

# Sans BUCKET_S3, stockage.py avertit qu'il lit et écrit dans data/ : c'est le cas prévu d'un
# compte neuf (README). Les autres avertissements, dont le repli d'une copie gelée, restent
warnings.filterwarnings("ignore", message="Variables d'environnement manquantes")
# pandas coupe les cases à 50 caractères : les intervalles et les sha256 seraient tronqués
pd.set_option("display.max_colwidth", None)

## Résumé

Dans les aires urbaines de Nantes, Lyon et Strasbourg, les communes proches d'un lycée privé ou d'un lycée à forte valeur ajoutée comptent-elles plus de cadres que leurs voisines ? Notre base de 789 communes croise le recensement, l'annuaire de l'éducation, la valeur ajoutée des lycées et quatre autres infrastructures.

Entre communes contiguës, être plus près d'un lycée privé ne va pas de pair avec un écart net de part de cadres. À Lyon, pour 5 km de moins jusqu'au lycée privé, l'intervalle va de −1,0 à +2,7 points de cadres : la règle fixée à l'avance parle de « résultat nul établi », mais cette borne est sans doute optimiste (4.5). À Nantes et à Strasbourg, les données ne permettent pas de conclure. La hausse des cadres entre 2011 et 2022 va davantage de pair avec le nombre de lycées proches qu'avec leur valeur ajoutée, selon un test qui désavantage la qualité (4.4).

Ce sont des associations : un lycée privé peut aussi s'implanter là où vivent déjà des cadres.

## 1. Introduction

### 1.1 Pourquoi les lycées

Choisir un logement, c'est souvent choisir une école. À Paris, Fack et Grenet (2010) trouvent qu'un écart-type de performance en plus pour le collège public de secteur va de pair avec des prix immobiliers plus élevés de 1,4 à 2,4 %, et que cet écart se réduit quand l'offre privée est proche : le privé sert d'option de repli face à la carte scolaire. Black (1999) compare des logements de part et d'autre d'une limite de secteur scolaire ; nos paires de communes contiguës en sont une version simplifiée.

Nous transposons la question aux lycées de province, dont la valeur ajoutée est publiée chaque année. Le poids du privé y varie fortement : environ 78 % des élèves du second degré sont scolarisés dans le public en métropole, contre six sur dix environ dans les académies de Nantes et de Rennes (DEPP, *Repères et références statistiques*). Dans l'Ouest, le privé est un choix courant, sans doute moins sélectif socialement. Comparer une aire de l'Ouest à deux autres permet de voir si le lien entre lycées et cadres en dépend.


### 1.2 Question et hypothèses

**Question.** À position égale dans l'aire, les communes qui offrent un meilleur accès à des lycées privés ou à forte valeur ajoutée comptent-elles plus de cadres que leurs voisines, et ce lien tient-il face aux autres infrastructures ? Les hypothèses ont été écrites avant de lire toute donnée sur les cadres.

| | Hypothèse | Comparaison principale | Statut |
|---|---|---|---|
| H1 | À distance égale du centre, plus de cadres là où l'accès aux lycées privés sous contrat est meilleur | Paires de communes contiguës (4.2), anneaux de distance (4.1) | verdict par aire |
| H2 | Ce lien est plus faible dans l'aire de l'Ouest (Nantes), où le privé est plus répandu | H1 aire par aire (4.3) | descriptive |
| H3 | La qualité des lycées accessibles, plus que leur nombre, accompagne la hausse de la part de cadres de 2011 à 2022 | Régression sur la qualité et le nombre (4.4) | verdict, 3 aires ensemble |
| H4 | Le lien est plus fort avec les lycées qu'avec les gares, les médecins et les équipements | Mêmes paires pour chaque infrastructure (5) | descriptive |

Les règles de verdict ont été fixées à l'avance et sont rappelées avec chaque résultat. H2 reste descriptive car une seule aire de l'Ouest ne permet pas de tester une différence ; H4, car aucune règle n'avait été fixée avant son estimation (2.5).


### 1.3 Ce que le projet peut et ne peut pas montrer

Le projet décrit des associations. Quatre limites empêchent d'aller plus loin :

- **causalité inverse** : un lycée privé peut s'implanter ou se maintenir là où vivent déjà des familles aisées ;
- **sélection résidentielle** : les préférences des ménages ne sont pas observées ;
- **croissance démographique** : elle peut précéder l'ouverture de lycées et accompagne l'arrivée de cadres ; elle n'est pas contrôlée ;
- **mesure de l'accès** : distances à vol d'oiseau depuis le centroïde de la commune.

Un résultat nul bien établi est un résultat : nous disons alors quelle taille d'association l'intervalle de confiance exclut.

## 2. Construire la base : une ligne par commune, quatre familles de variables

### 2.1 Schéma du pipeline et tableau des sources

Le rapport reconstruit la base à chaque exécution, depuis des fichiers bruts dont l'empreinte `sha256` est vérifiée (annexe). Chaque étape est une fonction de `src/cadres_lycees/` qui échoue si une jointure perd des lignes.

```{mermaid}
flowchart LR
    A[Communes<br/>aires, contours] --> D[Accessibilité<br/>une fonction pour tous les points]
    C[Lycées<br/>annuaire, IPS, IVAL] --> D
    E[Gares, médecins,<br/>sport, culture] --> D
    B[Recensement<br/>cadres 2011 et 2022] --> G[Base communale]
    D --> G
    F[Contrôles<br/>distance au centre, densité, prix] --> G
    G --> P[Paires de communes contiguës]
```

| Source | Producteur | Accès | Usage |
|---|---|---|---|
| Recensement 2022 (évolution et structure de la population, séries historiques) | Insee | fichiers | cadres 2011 et 2022, densité |
| Annuaire de l'éducation | Éducation nationale | API, copie gelée | lycées, secteur, coordonnées |
| IPS et IVAL des lycées | DEPP | API, copies gelées | composition sociale, valeur ajoutée |
| Aires d'attraction des villes, table de passage | Insee | fichiers | périmètre, codes communes |
| Contours des communes et des IRIS | IGN, `cartiflette` | copies gelées | géométries, contiguïté |
| Gares, APL, Data ES, BPE 2025 | SNCF, DREES, Sports, Insee | API et fichiers, copies gelées | H4 |
| Statistiques DVF | DGFiP, Etalab | copie gelée | prix au m² (robustesse) |

: Sources de données du projet {#tbl-sources}

In [ ]:
# Le rapport reconstruit la base depuis les copies gelées : final/ n'est
# qu'une sortie. Les fichiers lus et leur sha256 sont listés en annexe
base, lycees, paires = pipeline.construire_base(config.AIRES_RETENUES)

### 2.2 Variable à expliquer

Nous mesurons la part des cadres et professions intellectuelles supérieures parmi les actifs de 25 à 54 ans (occupés et chômeurs ayant déjà travaillé), en 2011 et en 2022, et son évolution en points. Cette tranche d'âge est celle des parents de lycéens et écarte les retraités. La source donne les arrondissements de Lyon, avec la même définition aux deux dates. H1, H2 et H4 portent sur 2022, H3 sur l'évolution.


### 2.3 Variable d'intérêt : l'accessibilité aux lycées

Nous mesurons l'accès, et non la présence d'un lycée dans la commune. Sont retenus les lycées généraux, technologiques et polyvalents, publics et privés sous contrat. Depuis le centroïde de chaque commune, à vol d'oiseau :

- la distance au lycée privé le plus proche (mesure principale de H1) ;
- le nombre de lycées privés à 5 km (mesure secondaire) ;
- l'IVAL moyenne des lycées à 5 km, pondérée par leurs candidats au bac. L'IVAL (indicateur de valeur ajoutée des lycées) compare le taux d'accès de la seconde au bac observé à celui qu'on attendrait au vu du profil des élèves ; nous en prenons la moyenne des sessions 2012 à 2022.

Les 2 413 lycées retenus de métropole comptent, y compris hors des aires, pour que le bord de l'aire ne fausse pas les distances.

In [ ]:
#| label: tbl-acces
#| tbl-cap: "Accès des communes aux lycées et aux autres infrastructures, médianes par aire."
# Accès des communes aux lycées et aux autres infrastructures, aire par aire (médianes)
tableaux.tableau_acces(analyse.resume_acces(base))

Le privé est bien moins accessible à Strasbourg : 4 % des communes y ont un lycée privé à 5 km, contre 16 % à Nantes et 23 % à Lyon, et la commune médiane en est à 16 km. Le lycée public est à 6 à 8 km en médiane dans les trois aires. À Strasbourg, le petit nombre de communes exposées limitera la précision.

### 2.4 Contrôles et mauvais contrôles

Les comparaisons se font à l'intérieur d'une aire, avec deux contrôles : la distance au centre de l'aire et le logarithme de la densité. La distance au lycée privé augmente avec la première et diminue avec la seconde ; sans ces contrôles, on retrouverait le gradient centre-périphérie (3.2).

Le niveau de vie, l'IPS des lycées (indice de position sociale, calculé à partir des professions des parents d'élèves) et la croissance de la population ne sont jamais des contrôles : ils dépendent en partie de la part de cadres. Le prix au m², lié à elle dans les deux sens, n'entre qu'en robustesse.


### 2.5 Encadré « choix et biais »

::: {.callout-note title="Choix et biais"}
- **Nomenclature.** Le recensement de 2022 code les professions en PCS 2020, celui de 2011 en PCS 2003. Selon l'Insee, les deux premiers niveaux, dont le groupe des cadres, sont inchangés ; les reclassements de professions détaillées ne sont pas vérifiés. Seule H3, qui compare 2011 et 2022, y est exposée.
- **Valeurs manquantes.** Aucune commune des trois aires n'est sans actif ni sans part de cadres. Ce qui manque est traité explicitement : IVAL absente pour 12 communes de la population de H3 (catégorie « sans IVAL »), prix au m² absent sous 10 ventes et dans toute l'Alsace-Moselle, 7 lycées hors contrat sans coordonnées, écartés de la robustesse qui ajoute le privé hors contrat. Toute autre donnée manquante fait échouer le code.
- **IVAL.** Mesure relative (elle situe un lycée par rapport à ce qu'on attend de ses élèves) et instable d'une session à l'autre, d'où la moyenne 2012-2022, qui couvre la période entre les deux recensements. Sa fiabilité au niveau des communes vaut 0,608 (@tbl-fiabilite) : au-dessus du seuil de 0,5 fixé à l'avance, H3 est testable.
- **Prix.** Les statistiques DVF couvrent juillet 2021 à juin 2026, après le recensement 2022 ; l'Alsace-Moselle en est absente. La robustesse au prix porte sur Lyon et Nantes seulement.
- **Lycées étrangers.** Absents de l'annuaire ; 55 communes de l'aire de Strasbourg sont à moins de 10 km de la frontière, et leur exclusion est testée.
- **Lycées agricoles.** Exclus partout, leur contrat avec le ministère de l'Agriculture n'étant pas couvert par nos règles ; la Loire-Atlantique en compte 15 dans le privé, une limite pour H2.
- **Vol d'oiseau depuis le centroïde.** Les distances depuis la mairie sont testées.
- **Lyon par arrondissements**, sauf dans la carte à curseur (3.2).
- **Petites communes.** Sous 200 actifs, la part de cadres, tirée d'un échantillon, est imprécise : 33 communes à Lyon, 72 à Strasbourg. Leur exclusion est testée.
- **Pré-enregistrement imparfait.** Hypothèses, choix des aires et règles de verdict étaient écrits avant les résultats, mais les pentes de H1 et H4 ont été vues le 03/11, avant la pose du tag Git censé figer le plan. Quelques détails d'exécution ont été fixés le même jour, après ces chiffres ; tout est daté dans le journal (annexe).
:::

## 3. Trois aires, trois profils

### 3.1 Pourquoi ces trois aires

Huit aires candidates ont été comparées sur six critères, sans aucune donnée sur les cadres (tableau de support ci-dessous, journal du 20/10). Paris est exclue pour le poids des transports en commun.

- **(a) Une aire de l'Ouest** : Nantes plutôt que Rennes : les deux ont une seule cellule d'anneaux exploitable (voir 4.1), et Nantes a plus de paires de voisines dont une seule a un lycée privé à 10 km (76 contre 71).
- **(b) Un contraste du privé** : 47 % des lycées à Nantes, 45 % à Lyon, 27 % à Strasbourg ; écart d'IPS entre privé et public de 11,8, 23,7 et 22,8 points.
- **(c) Des tailles comparables** : 1,04 million d'habitants à Nantes, 0,87 million à Strasbourg ; Lyon (2,33 millions) est la plus grande des huit.
- **(d) Une dizaine de lycées privés** : 18, 37 et 9 ; Strasbourg est juste sous le seuil.
- **(e) Un support suffisant** : Lyon est la seule aire où la comparaison par anneaux (4.1) est possible à 5 km ; à 10 km, elle l'est à Lyon, Nantes, Rennes et Strasbourg.
- **(f) La faisabilité** : la source donne les arrondissements de Lyon.

Strasbourg a été préférée à Bordeaux pour son contraste avec Nantes et ses anneaux comparables à 10 km. Sa fragilité était connue : 11 communes exposées à 5 km.

In [ ]:
#| column: page
#| label: fig-aires
#| fig-cap: >-
#|   Part du privé sous contrat parmi les lycées et les candidats au bac, et IPS de chaque
#|   lycée par secteur, dans les 8 aires candidates (aires retenues en gras).
# La `fig_aires` montre les 8 candidates pour justifier le choix ; la base du rapport (section 2) ne
# porte que sur les 3 aires retenues
referentiel_candidates = geo.referentiel_communes(config.AIRES_CANDIDATES)
lycees_8_aires = pipeline.lycees_avec_aire(referentiel_candidates)
profil = analyse.profil_aires(lycees_8_aires)
figures.fig_aires(
    profil,
    lycees_8_aires,
    "À Nantes, le privé est répandu et peu sélectif ; à Strasbourg, rare et sélectif",
    config.AIRES_RETENUES,
)

Le privé est aussi répandu à Nantes qu'à Lyon (47 % et 45 % des lycées), mais l'écart d'IPS entre privé et public y est deux fois plus faible (+12 contre +24 points). À Strasbourg, le privé est rare (16 % des candidats au bac) et aussi sélectif qu'à Lyon. La prémisse de H2 se vérifie : dans l'Ouest, le privé est plus répandu et moins sélectif.

In [ ]:
#| output: asis
# Le tableau de support du choix des aires, commité avant le choix (commit 0583839),
# replié : il justifie le choix sans alourdir la lecture
print(
    tableaux.encadre_replie(
        "Tableau de support du choix des aires",
        tableaux.document_markdown(config.CHEMIN_TABLEAU_SUPPORT),
    )
)

### 3.2 Le gradient centre-périphérie des cadres

La part de cadres baisse avec la distance au centre. Comparer des communes proches et éloignées d'un lycée privé sans tenir compte de leur position reviendrait donc à comparer le centre et la périphérie.

In [ ]:
#| label: tbl-gradient
#| tbl-cap: "Part de cadres des 25-54 ans en 2022 par anneau de distance au centre."
# Part de cadres de chaque anneau de distance au centre en 2022 (cadres sur actifs des communes
# de l'anneau), aire par aire : le gradient qui justifie de comparer à distance égale
tableaux.tableau_gradient(analyse.parts_par_anneau(base))

La part de cadres est 1,9 à 2,5 fois plus élevée à moins de 5 km du centre qu'au-delà de 30 km : 39 % contre 17 % à Lyon, 36 % contre 15 % à Nantes, 29 % contre 16 % à Strasbourg. À Lyon, l'anneau de 5 à 10 km (23 %) passe sous celui de 10 à 20 km (26 %). À Nantes et à Strasbourg, l'anneau 0-5 km ne compte que 2 communes : sa valeur est celle de la commune-centre et d'une voisine.

In [ ]:
#| output: asis
# Part de cadres de 1999 à 2022, de la série historique du recensement ramenée au COG 2025 ;
# une aire par onglet, comme la `carte_acces`, et un curseur pour choisir le recensement
series_csp = {annee: collecte.serie_csp_2554(annee) for annee in config.GEOGRAPHIE_SERIE_CSP}
parts_dates = nettoyage.parts_cadres_dates(series_csp, collecte.passage_communes(), base)
cartes_dates = {
    aire: cartes.carte_cadres_dates(base[base["aire"] == aire], parts_dates, f"Aire de {aire}")
    for aire in config.AIRES_RETENUES
}
legende = (
    "**La part de cadres baisse du centre vers la périphérie à chaque recensement "
    "depuis 1999.** Part des cadres parmi les actifs de 25 à 54 ans par commune, de"
    " 1999 à 2022, une aire par onglet et un curseur pour l'année."
)
for morceau in cartes.onglets_par_aire("fig-cadres-dates", cartes_dates, legende):
    # Le balisage Quarto est imprimé, les cartes sont affichées
    if isinstance(morceau, str):
        print(morceau)
    else:
        display(morceau)

Ce gradient se retrouve à chaque recensement depuis 1999, tandis que la part de cadres monte dans les trois aires. La carte ne montre pas les lycées et ne dit rien de H1.

### 3.3 La géographie de l'accès aux lycées

In [ ]:
#| output: asis
# Une carte par aire retenue, chacune dans un onglet
cartes_acces = {
    aire: cartes.carte_acces(base[base["aire"] == aire], lycees, f"Aire de {aire}")
    for aire in config.AIRES_RETENUES
}
legende = (
    "**Hors des villes-centres, le lycée privé le plus proche est souvent à plus de"
    " 10 km.** Distance à vol d'oiseau du centroïde de chaque commune au lycée "
    "privé sous contrat le plus proche, lycées publics et privés en points, une "
    "aire par onglet."
)
for morceau in cartes.onglets_par_aire("fig-acces", cartes_acces, legende):
    # Le balisage Quarto est imprimé, les cartes sont affichées
    if isinstance(morceau, str):
        print(morceau)
    else:
        display(morceau)

Les lycées privés sont regroupés dans les villes-centres et quelques pôles secondaires. Au-delà, la distance augmente vite, surtout à Strasbourg, où la plupart des communes sont à plus de 10 km d'un lycée privé. Les communes exposées sont donc plus centrales que les autres : la section 4 compare à position égale.

## 4. À position égale, un meilleur accès aux lycées va-t-il de pair avec plus de cadres ?

### 4.1 Comparaison par anneaux

La figure ci-dessous rassemble toutes les estimations de H1. Ses deux premiers panneaux donnent les pentes des paires de communes contiguës, détaillées en 4.2 ; le troisième, les anneaux, décrits ici.

Dans chaque anneau de distance au centre (0-5, 5-10, 10-20, 20-30, plus de 30 km), on compare les communes qui ont un lycée privé dans le rayon (exposées) aux autres, avec les contrôles de 2.4. Une cellule (aire × anneau) n'est estimée que si chaque groupe compte au moins 15 communes et si leurs distances moyennes au centre diffèrent de 2 km au plus ; sinon, on fusionne des anneaux voisins. Les intervalles viennent d'un bootstrap : 1 000 tirages des communes avec remise. Seules 6 cellules passent ces règles : 2 à Lyon à 5 km ; à 10 km, 2 à Lyon, 1 à Nantes, 1 à Strasbourg (panneau de droite).

In [ ]:
#| column: page
#| label: fig-h1
#| fig-cap: >-
#|   Estimations de H1 par aire, avec intervalles à 95 % : pentes principale et secondaire des
#|   paires contiguës, écarts des cellules d'anneaux estimables à 5 et 10 km.
# H1 : pentes des paires, principale sur l'écart de distance au lycée privé et secondaire sur
# l'écart du nombre de lycées privés à 5 km, avec les mêmes tirages du bootstrap ; écarts
# des cellules d'anneaux à 5 et 10 km. La `fig_h1` sert aussi H2
pentes_principales = analyse.pentes_paires_par_aire(paires, "d_dist_lyc_prive_min")
pentes_secondaires = analyse.pentes_paires_par_aire(paires, "d_n_lyc_prive_5km")
anneaux = analyse.ecarts_anneaux(base)
figures.fig_h1(
    pentes_principales,
    pentes_secondaires,
    anneaux,
    "Aucune pente des paires n'exclut zéro ; une seule cellule d'anneau l'exclut",
)

Les quatre cellules de Lyon vont dans le sens de H1 (+1,5 à +3,1 points de cadres dans les communes exposées), mais aucun intervalle n'exclut zéro ; elles concordent entre les deux rayons. Seule la cellule de Nantes au-delà de 20 km exclut zéro : +2,0 points [+0,1 ; +3,9]. Les anneaux ne tranchent pas seuls ; ils servent au verdict des paires.

### 4.2 Comparaison entre communes contiguës

Le résultat principal compare des **paires contiguës**, deux communes de la même aire qui ont une frontière ou un point commun : 301 paires à Nantes, 1 127 à Lyon, 717 à Strasbourg. Deux voisines partagent marché de l'emploi et position dans l'aire, qui s'annulent dans la différence. On régresse l'écart de part de cadres sur l'écart de distance au lycée privé le plus proche, avec les écarts de distance au centre et de densité. H1 attend une pente négative.

Une commune entre dans plusieurs paires : les intervalles viennent d'un **bootstrap par commune**, qui tire les communes et non les paires (1 000 tirages).

In [ ]:
#| column: page
#| label: fig-paires
#| fig-cap: >-
#|   Écart de part de cadres en 2022 et écart de distance au lycée privé le plus proche entre
#|   communes contiguës, à position égale, moyennes par classes et pente principale.
figures.fig_paires(
    paires,
    pentes_principales,
    "Entre communes voisines, la pente est faible et son intervalle contient zéro partout",
)

La pente est négative dans les trois aires, mais faible : −0,15 point de cadres par km à Nantes, −0,18 à Lyon, −0,06 à Strasbourg, et les trois intervalles contiennent zéro. À Lyon, l'intervalle [−0,53 ; +0,21] signifie qu'être 5 km plus près d'un lycée privé s'accompagne d'un écart compris entre −1,0 et +2,7 points de cadres. C'est peu à côté de l'écart de 22 points entre le centre et la périphérie.

Le nuage est symétrique autour de zéro par construction : chaque paire y figure dans les deux sens, puisque l'ordre des deux communes est arbitraire.

Verdicts, selon les règles fixées à l'avance :

- **Lyon : « infirmée » (résultat nul établi).** L'intervalle contient zéro, les deux anneaux comparés aux rayons de 5 et 10 km concordent, et aucune cellule n'exclut zéro. Toutes les estimations de Lyon vont pourtant dans le sens de H1 (pente négative, pente secondaire et cellules positives), sans qu'aucune n'exclue zéro : « infirmée » est le nom, fixé à l'avance, d'un lien trop faible pour être distingué de zéro, pas d'un lien de sens contraire.
- **Nantes et Strasbourg : « non concluante ».** L'intervalle contient zéro, mais sans anneau estimable aux deux rayons, la règle interdit de parler de résultat nul établi.

La pente secondaire, sur le nombre de lycées privés à 5 km, ne change rien : son intervalle contient zéro partout (@fig-h1).

### 4.3 L'Ouest est-il différent ?

Les pentes principales des trois aires sont très proches (@fig-h1) et leurs intervalles se recouvrent largement : rien ne distingue Nantes. La seule estimation qui exclut zéro, la cellule nantaise au-delà de 20 km, irait même à l'inverse de H2, mais elle est isolée. Un lien faible partout laisse peu de place à des écarts entre aires ; H2 reste descriptive.

### 4.4 Qualité ou présence ?

H3 porte sur l'évolution de la part de cadres de 2011 à 2022, dans les 280 communes qui ont un lycée retenu à 5 km ; l'estimation porte sur les 268 d'entre elles dont au moins un de ces lycées a une IVAL. On la régresse sur deux mesures centrées réduites dans chaque aire, avec les contrôles de 2.4, à l'intérieur de chaque aire et de chaque anneau : **[β~q~]{.grec}** est la pente de l'IVAL moyenne (la qualité), **[β~n~]{.grec}** celle du logarithme du nombre de lycées à 5 km qui ont une IVAL (la présence, mesurée sur les mêmes lycées que la qualité). Les intervalles viennent d'un bootstrap par grappes de communes qui ont le même lycée le plus proche. H3 est « infirmée » si l'intervalle de [β~q~]{.grec} ou de [β~q~]{.grec} − [β~n~]{.grec} est entièrement sous zéro.

In [ ]:
#| label: tbl-fiabilite
#| tbl-cap: "Fiabilité de l'IVAL à 5 km : moitiés 2012-2017 et 2018-2022, Spearman-Brown."
# Fiabilité de l'IVAL : seule la ligne « ensemble » tranche H3 (seuil de 0,5)
fiabilite = analyse.fiabilite_ival(base, lycees, collecte.ival_lycees())
tableaux.tableau_fiabilite(fiabilite)

La fiabilité d'ensemble, 0,608, dépasse le seuil de 0,5 : H3 est testable. Elle est portée par Lyon (0,657) ; à Nantes et à Strasbourg, elle reste sous 0,5.

In [ ]:
#| label: fig-h3
#| fig-cap: >-
#|   Évolution moyenne de la part de cadres de 2011 à 2022 par tiers d'IVAL des lycées à 5 km,
#|   calculés dans chaque aire, communes sans IVAL à part.
# H3 : population des communes ayant un lycée retenu à 5 km, régression sur l'IVAL
# et le nombre de lycées à IVAL, centrés réduits dans chaque aire, bootstrap par grappes
population_h3 = analyse.population_h3(base, lycees)
estimation_h3 = analyse.estimation_h3(population_h3)
figures.fig_h3(
    analyse.terciles_h3(population_h3),
    "La hausse de la part de cadres ne suit pas régulièrement l'IVAL des lycées proches",
)

In [ ]:
#| label: tbl-h3
#| tbl-cap: "Coefficients de H3, en points pour un écart-type, intervalles à 95 %."
# β en points de part de cadres pour un écart-type ; seule la portée « ensemble » porte le verdict
tableaux.tableau_h3(estimation_h3)

La hausse ne progresse pas régulièrement avec l'IVAL : elle baisse légèrement d'un tiers à l'autre à Lyon (5,4 ; 4,6 puis 4,3 points), monte à Strasbourg avec des intervalles très larges, et n'a pas d'ordre à Nantes.

Dans @tbl-h3, un écart-type d'IVAL en plus va de pair avec +0,22 point de hausse [−0,48 ; +0,99], un écart-type de nombre de lycées avec +1,73 point [+1,13 ; +2,80]. La différence [β~q~]{.grec} − [β~n~]{.grec} vaut −1,51 [−2,85 ; −0,63] : entièrement sous zéro, elle rend H3 **« infirmée »**. Les colonnes par aire sont descriptives : à Nantes et à Strasbourg, la fiabilité de l'IVAL est sous 0,5 (@tbl-fiabilite), et leurs intervalles sont très larges.

Ce test penche structurellement contre H3. L'erreur de mesure de l'IVAL tire [β~q~]{.grec} vers zéro, pas [β~n~]{.grec}. Et le stock de lycées est celui de 2026 : des lycées ouverts après 2011 dans des communes en croissance gonflent [β~n~]{.grec}. Le verdict tient avec d'autres mesures de la qualité (4.5), mais ces variantes ne corrigent ni l'atténuation de [β~q~]{.grec} ni le stock de lycées de 2026 ; l'autocorrélation spatiale (4.5) rend aussi l'intervalle de [β~q~ − β~n~]{.grec} sans doute trop étroit. La hausse des cadres va donc de pair avec la densité de l'offre de lycées plus qu'avec sa valeur ajoutée mesurée, sans qu'on sache laquelle précède l'autre.

### 4.5 Robustesse : tableau récapitulatif et indice de Moran

Une seule spécification porte chaque verdict ; les variantes vérifient qu'il ne tient pas à un choix particulier.

#### Pente principale de H1

In [ ]:
#| label: fig-robustesse
#| fig-cap: "Pente principale de H1 dans chaque variante de robustesse, avec intervalles à 95 %."
# Robustesse : pente de H1 dans chaque variante, avec les mêmes estimation et bootstrap
# que la spécification principale
robustesse = analyse.robustesse_h1(
    base, paires, lycees, pipeline.lycees_prives_robustesse(), collecte.chefs_lieux()
)
figures.fig_robustesse(
    robustesse, "Aucune variante ne fait sortir zéro de l'intervalle de la pente principale"
)

Aucune variante ne fait sortir zéro de l'intervalle de la pente principale. Le contrôle de densité est le choix qui pèse le plus : sans lui, la pente de Lyon passe de −0,18 à −0,01, celle de Nantes de −0,15 à −0,29 [−0,67 ; +0,12]. Dans les deux aires, les communes plus denses sont plus proches d'un lycée privé. À distance égale, elles comptent un peu moins de cadres à Lyon et un peu plus à Nantes : omettre la densité ramène donc la pente de Lyon vers zéro et rend celle de Nantes plus négative. C'est pourquoi la densité est contrôlée dans la spécification principale (2.4). Ajouter les lycées professionnels et le privé hors contrat donne les pentes les plus négatives (Lyon −0,31, Strasbourg −0,24), avec des intervalles qui contiennent encore zéro. La pente secondaire de Lyon change de signe entre 3 et 10 km : elle n'apporte rien de stable.

In [ ]:
#| output: asis
# Le détail, pentes secondaires comprises, replié sous la figure
detail = tableaux.tableau_robustesse(robustesse).to_markdown()
legende = ": Pente de H1 dans chaque variante, en points, intervalle à 95 % {#tbl-robustesse}"
print(tableaux.encadre_replie("Tableau de robustesse détaillé", f"{detail}\n\n{legende}"))

#### Qualité des lycées (H3)

In [ ]:
#| label: tbl-h3-robustesse
#| tbl-cap: "Coefficients de H3 selon la période et l'indicateur de l'IVAL, 3 aires ensemble."
# Robustesse de H3 : période de l'IVAL et indicateur de valeur ajoutée, 3 aires ensemble
tableaux.tableau_h3(analyse.robustesse_h3(base, lycees, collecte.ival_lycees()))

Le verdict tient avec l'IVAL de 2012 à 2025 ([β~q~]{.grec} − [β~n~]{.grec} = −1,39 [−2,63 ; −0,35]) et avec la valeur ajoutée du taux de réussite (−1,91 [−3,45 ; −0,75]).

#### Autocorrélation spatiale

L'indice de Moran mesure si des communes voisines se ressemblent plus que par hasard : proche de 0 sans dépendance spatiale, il monte vers 1 quand les voisines se ressemblent.

In [ ]:
#| label: tbl-moran
#| tbl-cap: "Indice I de Moran de la part de cadres et de l'écart à la moyenne de l'anneau."
# Indice I de Moran de la part de cadres, puis de l'écart à la moyenne de son anneau
tableaux.tableau_moran(analyse.moran_part_cadres(base))

La part de cadres est très concentrée dans l'espace (0,33 à 0,61). Une fois retirée la moyenne de chaque anneau, une dépendance nette demeure, surtout à Lyon (0,31). Deux voisines ne sont donc pas deux observations indépendantes, et le bootstrap par commune n'en tient compte qu'en partie : nos intervalles sont probablement trop étroits. Les intervalles qui contiennent zéro le contiendraient encore s'ils étaient élargis ; en revanche, les tailles d'association qu'ils excluent sont moins sûres qu'affiché, et les intervalles de H3 sont à lire avec prudence.

## 5. Les lycées face aux autres infrastructures

H4 compare le lycée privé à quatre infrastructures, avec les paires, contrôles et tirages de H1 : la gare la plus proche, l'APL aux médecins généralistes (accessibilité potentielle localisée, en consultations accessibles par habitant, calculée par la DREES), et le nombre d'équipements sportifs et culturels à 5 km. Chaque mesure est centrée réduite dans l'aire, orientée pour qu'une pente positive se lise « plus d'accès, plus de cadres ».

In [ ]:
#| label: fig-h4
#| fig-cap: >-
#|   Pente des paires contiguës pour un écart-type d'accès en plus, par infrastructure et par
#|   aire, avec intervalles à 95 % par bootstrap par commune.
# H4 : pentes des paires pour chaque mesure d'accès, centrée réduite dans chaque aire
pentes_h4 = analyse.pentes_h4(base, paires)
figures.fig_h4(
    pentes_h4, "Lycées, gares, médecins, sport, culture : aucune infrastructure ne se détache"
)

Les quinze pentes sont petites, avec des intervalles larges qui se recouvrent tous. Les pentes du lycée privé vont de +0,43 à +0,90 point par écart-type selon l'aire, sans qu'aucun intervalle n'exclue zéro ; d'autres pentes isolées sont plus fortes (installations sportives à Lyon, gare à Nantes). Seule la gare à Nantes exclut zéro (+1,64 [+0,12 ; +3,25]) : une estimation sur quinze, à peu près ce que produirait le hasard. Faute de règle fixée avant l'estimation, nous ne concluons pas que le lien est plus fort avec les lycées : aucune infrastructure ne se détache.

## 6. Conclusion

In [ ]:
#| label: tbl-hypotheses
#| tbl-cap: "Verdict de chaque hypothèse selon les règles fixées avant les résultats."
# Tableau des hypothèses, par les règles fixées avant les résultats : H1 par aire, H3 toutes
# aires ensemble ; H2 et H4 descriptives
verdicts_h1 = analyse.verdict_h1(pentes_principales, pentes_secondaires, anneaux)
verdict_h3 = analyse.verdict_h3(estimation_h3, fiabilite.loc["ensemble", "r_spearman_brown"])
tableaux.tableau_hypotheses(verdicts_h1, verdict_h3)

Entre communes voisines, être plus près d'un lycée privé ne va pas de pair avec un écart net de cadres : le résultat nul est établi à Lyon, non concluant à Nantes et à Strasbourg (H1), sans différence visible pour l'Ouest (H2) ; à Lyon, la borne du résultat nul est sans doute optimiste. La hausse des cadres va davantage de pair avec le nombre de lycées proches qu'avec leur valeur ajoutée (H3 infirmée), et aucune infrastructure ne se détache (H4).

Notre lecture : à l'échelle de la commune, une fois la position dans l'aire prise en compte, l'accès au lycée privé ne distingue pas les communes riches en cadres de leurs voisines. La géographie des cadres est d'abord une affaire de distance au centre (@tbl-gradient). Cela n'exclut pas que des familles choisissent leur logement selon le lycée : un lycée privé dessert plusieurs communes voisines à la fois, et la commune est sans doute une maille trop large pour voir ce choix.

Les limites sont celles de l'introduction (causalité inverse, sélection, croissance démographique, vol d'oiseau), auxquelles s'ajoutent la fragilité de Strasbourg et des intervalles probablement optimistes. La carte ci-dessous, exploratoire, illustre la question de la maille.

In [ ]:
#| output: asis
# Limites : part de cadres de 15 à 64 ans par IRIS en 2011 et 2022, exploration sans test
# ni lycées. Les communes-centres sont à 0 km du centre ; à Lyon, l'arrondissement le plus proche
# du centre a pour commune parente 69123, le code des contours IRIS. Une aire par onglet, comme
# la `carte_acces`
parts_iris = nettoyage.part_cadres_iris(
    collecte.activite_iris_2011(), collecte.activite_iris_2022(), collecte.passage_communes()
)
cartes_iris = {}
for aire, code_aav in config.AIRES_RETENUES.items():
    communes_aire = base[base["aire"] == aire]
    commune_centre = communes_aire.loc[communes_aire["dist_centre_km"].idxmin(), "code_parent"]
    contours = collecte.contours_iris(code_aav)
    cartes_iris[aire] = figures.carte_iris(contours, parts_iris, commune_centre, f"Aire de {aire}")
legende = (
    "**Dans les villes-centres, la part de cadres va de moins de 10 % à plus de 50 "
    "% selon l'IRIS.** Part des cadres parmi les actifs de 15 à 64 ans par IRIS en "
    "2011 et 2022, sur l'aire entière et autour de la commune-centre, une aire par "
    "onglet."
)
for morceau in cartes.onglets_par_aire("fig-iris", cartes_iris, legende):
    # Le balisage Quarto est imprimé, les cartes sont affichées
    if isinstance(morceau, str):
        print(morceau)
    else:
        display(morceau)

À l'intérieur des villes-centres, des IRIS de moins de 10 % de cadres côtoient des IRIS de 50 % ou plus, et les plus foncés se multiplient entre 2011 et 2022. Une moyenne communale mêle donc des quartiers très différents (la carte porte sur les 15-64 ans, seule tranche disponible par IRIS).

Quatre pistes prolongeraient ce travail :

- **l'échelle IRIS**, avec la même méthode de paires ;
- **les flux migratoires** (MIGCOM), pour voir où s'installent les cadres qui arrivent, ce qui répondrait en partie à la causalité inverse ;
- **les temps de trajet**, à la place du vol d'oiseau ;
- **les collèges**, au cœur du mécanisme de Fack et Grenet : à 5 km, ils donneraient environ deux fois plus de paires de voisines de statut différent (279 à Lyon, 113 à Nantes, 53 à Strasbourg, contre 169, 62 et 26 pour les lycées), et l'IPS des collèges comme la carte scolaire des collèges publics sont en open data.

## 7. Annexes

### Références

- Black, S. E. (1999). « Do Better Schools Matter? Parental Valuation of Elementary Education ». *Quarterly Journal of Economics*, 114(2), p. 577-599.
- Fack, G. et Grenet, J. (2010). « When do better schools raise housing prices? Evidence from Paris public and private schools ». *Journal of Public Economics*, 94(1-2), p. 59-77.
- DEPP, *Repères et références statistiques sur les enseignements, la formation et la recherche*, ministère de l'Éducation nationale, édition 2025.
- Insee, recensements de la population 1999 à 2022 ; DEPP, IPS et IVAL des lycées ; sources détaillées en 2.1 et dans l'annexe ci-dessous.

### Reproduire le projet

Il faut Python 3.13, [uv](https://docs.astral.sh/uv/) et [Quarto](https://quarto.org/) (déjà installé sur le SSPCloud), sans variable d'environnement.

```bash
uv sync --locked      # exactement les dépendances figées dans uv.lock
uv run quarto render  # exécute rapport.ipynb et écrit le site dans _site/
```

Les sources qui changent sans prévenir sont lues dans des copies gelées publiques (release GitHub du dépôt ; un dossier public du SSPCloud est prévu en secours, mais il n'est pas lisible aujourd'hui), les fichiers de l'Insee à leur URL d'origine. Le `sha256` de chaque fichier, listé dans @tbl-fichiers, est vérifié à chaque lecture. Sur un clone neuf, sans identifiants S3, l'exécution complète prend environ 12 minutes, téléchargements compris (mesuré le 03/11/2026).

In [ ]:
#| label: tbl-fichiers
#| tbl-cap: "Fichiers bruts lus par le rapport et leur sha256, contrôlé à chaque lecture."
pipeline.fichiers_bruts()

### Dictionnaire des variables

In [ ]:
#| output: asis
# Replié comme le journal : il sert de référence, pas de lecture suivie
print(
    tableaux.encadre_replie(
        "Dictionnaire des variables", tableaux.document_markdown(config.CHEMIN_DICTIONNAIRE)
    )
)

### Journal des décisions

In [ ]:
#| output: asis
# Le journal du plan, daté, tel qu'il est dans le dépôt ; replié, car il est plus long que
# le reste du rapport
print(
    tableaux.encadre_replie(
        "Journal des décisions",
        tableaux.document_markdown(config.CHEMIN_PLAN, config.TITRE_JOURNAL_PLAN),
    )
)